<a href="https://colab.research.google.com/github/viethaa/f1-big-data-analytics/blob/main/formula1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1><b>A 2025 Performance and Strategy Analysis of Formula 1's Top Constructors</b></h1>

Lorem ipsum dolor sit amet, consectetur adipiscing elit. Donec risus odio, blandit et sapien in, pharetra convallis nulla. Cras facilisis, arcu vitae sollicitudin ornare, velit arcu lacinia quam, vitae consequat mi felis quis dolor. Donec tincidunt commodo lectus nec porta. Pellentesque eu felis erat. Cras ac porta turpis. Quisque tincidunt auctor mattis. Maecenas convallis orci mauris, tempor rutrum nibh sodales sed. Curabitur in enim a leo tempus rutrum id vitae mauris. Quisque ullamcorper posuere fringilla. Morbi et ante lacinia justo dignissim placerat. The project repository can be found on github: https://github.com/viethaa/f1-big-data-analytics

# Install and Load Data

In [2]:
# Install packages
!pip install -q plotly

In [3]:
# Import libraries
import os
import requests
import time
import base64
from io import BytesIO

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from PIL import Image
from google.colab import drive

In [4]:
# Mount Drive
drive.mount('/content/drive')

YEAR = 2025
DATA_DIR = f"/content/drive/MyDrive/Big Data 2 - Ha, Bach Viet (Bob)/f1_data_{YEAR}"
JOLPICA = "https://api.jolpi.ca/ergast/f1"
OPENF1 = "https://api.openf1.org/v1"
MARKER = os.path.join(DATA_DIR, ".download_complete")
os.makedirs(DATA_DIR, exist_ok=True)

TEAM_COLORS = {
    "Ferrari": "#E8002D",
    "McLaren": "#FF8000",
    "Mercedes": "#27F4D2",
    "Red Bull": "#3671C6",
}

COMPOUND_COLORS = {
    "SOFT": "rgb(255, 45, 45)",
    "MEDIUM": "rgb(255, 210, 50)",
    "HARD": "rgb(230, 230, 230)",
    "INTERMEDIATE": "rgb(60, 180, 60)",
    "WET": "rgb(50, 130, 255)",
}

Mounted at /content/drive


In [5]:
# Download all data (skips files that already exist)

def api_get(url, params=None):
    for attempt in range(3):
        try:
            resp = requests.get(url, params=params, timeout=30)
            resp.raise_for_status()
            return resp.json()
        except:
            time.sleep(2)
    return None

def already_done(filename):
    path = f"{DATA_DIR}/{filename}"
    if os.path.exists(path):
        rows = len(pd.read_csv(path))
        print(f"   Already exists ({rows} rows) — skipping")
        return True
    return False

races = api_get(f"{JOLPICA}/{YEAR}.json")["MRData"]["RaceTable"]["Races"]

# 1. Schedule
print("1. Schedule...")
if not already_done(f"schedule_{YEAR}.csv"):
    rows = [{"round": int(r["round"]), "raceName": r["raceName"],
             "circuitId": r["Circuit"]["circuitId"], "circuitName": r["Circuit"]["circuitName"],
             "country": r["Circuit"]["Location"]["country"], "locality": r["Circuit"]["Location"]["locality"],
             "lat": r["Circuit"]["Location"]["lat"], "long": r["Circuit"]["Location"]["long"],
             "date": r["date"], "time": r.get("time", "")} for r in races]
    pd.DataFrame(rows).to_csv(f"{DATA_DIR}/schedule_{YEAR}.csv", index=False)
    print(f"   Done: {len(rows)} races")
time.sleep(0.5)

# 2. Driver standings
print("2. Driver standings...")
if not already_done(f"driver_standings_{YEAR}.csv"):
    ds = api_get(f"{JOLPICA}/{YEAR}/driverStandings.json")
    sl = ds["MRData"]["StandingsTable"]["StandingsLists"]
    if sl:
        rows = [{"position": int(s["position"]), "driverId": s["Driver"]["driverId"],
                 "driverCode": s["Driver"].get("code",""), "givenName": s["Driver"]["givenName"],
                 "familyName": s["Driver"]["familyName"], "nationality": s["Driver"]["nationality"],
                 "constructorId": s["Constructors"][0]["constructorId"] if s["Constructors"] else "",
                 "constructorName": s["Constructors"][0]["name"] if s["Constructors"] else "",
                 "points": float(s["points"]), "wins": int(s["wins"])} for s in sl[0]["DriverStandings"]]
        pd.DataFrame(rows).to_csv(f"{DATA_DIR}/driver_standings_{YEAR}.csv", index=False)
        print(f"   Done: {len(rows)} drivers")
time.sleep(0.5)

# 3. Constructor standings
print("3. Constructor standings...")
if not already_done(f"constructor_standings_{YEAR}.csv"):
    cs = api_get(f"{JOLPICA}/{YEAR}/constructorStandings.json")
    cl = cs["MRData"]["StandingsTable"]["StandingsLists"]
    if cl:
        rows = [{"position": int(s["position"]), "constructorId": s["Constructor"]["constructorId"],
                 "constructorName": s["Constructor"]["name"], "nationality": s["Constructor"]["nationality"],
                 "points": float(s["points"]), "wins": int(s["wins"])} for s in cl[0]["ConstructorStandings"]]
        pd.DataFrame(rows).to_csv(f"{DATA_DIR}/constructor_standings_{YEAR}.csv", index=False)
        print(f"   Done: {len(rows)} constructors")
time.sleep(0.5)

# 4. Race results
print("4. Race results...")
if not already_done(f"results_{YEAR}.csv"):
    all_results = []
    for race in races:
        rnd, name = race["round"], race["raceName"]
        try:
            resp = api_get(f"{JOLPICA}/{YEAR}/{rnd}/results.json", params={"limit": 30})
            rd = resp["MRData"]["RaceTable"]["Races"]
            if rd and "Results" in rd[0]:
                for r in rd[0]["Results"]:
                    fl = r.get("FastestLap") or {}
                    all_results.append({
                        "round": int(rnd), "race_name": name,
                        "driverId": r["Driver"]["driverId"], "driverCode": r["Driver"].get("code",""),
                        "constructorId": r["Constructor"]["constructorId"], "constructorName": r["Constructor"]["name"],
                        "grid": int(r["grid"]), "position": r.get("position"),
                        "positionText": r.get("positionText",""), "status": r.get("status",""),
                        "points": float(r.get("points",0)), "laps": int(r.get("laps",0)),
                        "time_text": r.get("Time",{}).get("time","") if isinstance(r.get("Time"),dict) else "",
                        "fastest_lap_rank": fl.get("rank",""),
                        "fastest_lap_time": fl.get("Time",{}).get("time","") if fl.get("Time") else "",
                        "fastest_lap_speed_kph": fl.get("AverageSpeed",{}).get("speed","") if fl.get("AverageSpeed") else "",
                    })
        except: pass
        time.sleep(0.5)
    pd.DataFrame(all_results).to_csv(f"{DATA_DIR}/results_{YEAR}.csv", index=False)
    print(f"   Done: {len(all_results)}")
time.sleep(0.5)

# 5. Qualifying
print("5. Qualifying...")
if not already_done(f"qualifying_{YEAR}.csv"):
    all_quali = []
    for race in races:
        rnd, name = race["round"], race["raceName"]
        try:
            resp = api_get(f"{JOLPICA}/{YEAR}/{rnd}/qualifying.json", params={"limit": 30})
            rd = resp["MRData"]["RaceTable"]["Races"]
            if rd and "QualifyingResults" in rd[0]:
                for q in rd[0]["QualifyingResults"]:
                    all_quali.append({
                        "round": int(rnd), "race_name": name,
                        "driverId": q["Driver"]["driverId"], "driverCode": q["Driver"].get("code",""),
                        "constructorId": q["Constructor"]["constructorId"],
                        "position": int(q["position"]), "Q1": q.get("Q1",""), "Q2": q.get("Q2",""), "Q3": q.get("Q3",""),
                    })
        except: pass
        time.sleep(0.5)
    pd.DataFrame(all_quali).to_csv(f"{DATA_DIR}/qualifying_{YEAR}.csv", index=False)
    print(f"   Done: {len(all_quali)}")
time.sleep(0.5)

# 6. Pit stops
print("6. Pit stops...")
if not already_done(f"pitstops_{YEAR}.csv"):
    all_pits = []
    for race in races:
        rnd, name = race["round"], race["raceName"]
        try:
            resp = api_get(f"{JOLPICA}/{YEAR}/{rnd}/pitstops.json", params={"limit": 200})
            rd = resp["MRData"]["RaceTable"]["Races"]
            if rd and "PitStops" in rd[0]:
                for s in rd[0]["PitStops"]:
                    dur = s.get("duration","")
                    sec = float(dur.split(":")[0])*60 + float(dur.split(":")[1]) if ":" in str(dur) else (float(dur) if dur else None)
                    all_pits.append({"round": int(rnd), "race_name": name, "driverId": s["driverId"],
                                     "stop": int(s["stop"]), "lap": int(s["lap"]),
                                     "time_of_day": s.get("time",""), "duration_raw": dur, "duration_sec": sec})
        except: pass
        time.sleep(0.5)
    pd.DataFrame(all_pits).to_csv(f"{DATA_DIR}/pitstops_{YEAR}.csv", index=False)
    print(f"   Done: {len(all_pits)}")
time.sleep(0.5)

# 7. Lap times
print("7. Lap times (slowest step — ~5 min)...")
if not already_done(f"laptimes_{YEAR}.csv"):
    all_laps = []
    for race in races:
        rnd, name = race["round"], race["raceName"]
        print(f"   [R{rnd}] {name}...", end=" ", flush=True)
        count = 0
        try:
            offset = 0
            while True:
                resp = api_get(f"{JOLPICA}/{YEAR}/{rnd}/laps.json", params={"limit": 1000, "offset": offset})
                rd = resp["MRData"]["RaceTable"]["Races"]
                if not rd or "Laps" not in rd[0] or not rd[0]["Laps"]: break
                for lap in rd[0]["Laps"]:
                    for t in lap["Timings"]:
                        lt = t.get("time","")
                        lt_sec = float(lt.split(":")[0])*60 + float(lt.split(":")[1]) if lt and ":" in lt else None
                        all_laps.append({"round": int(rnd), "race_name": name, "driverId": t["driverId"],
                                         "lap": int(lap["number"]), "position": int(t.get("position",0)),
                                         "time_text": lt, "time_sec": lt_sec})
                        count += 1
                offset += 1000
                if offset >= int(resp["MRData"]["total"]): break
                time.sleep(0.3)
            print(f"done ({count})")
        except: print(f"failed ({count} collected)")
        time.sleep(0.5)
    pd.DataFrame(all_laps).to_csv(f"{DATA_DIR}/laptimes_{YEAR}.csv", index=False)
    print(f"   Total: {len(all_laps)}")
time.sleep(0.5)

# 8. Sprints
print("8. Sprints...")
if not already_done(f"sprints_{YEAR}.csv"):
    all_sprints = []
    for race in races:
        rnd, name = race["round"], race["raceName"]
        try:
            resp = api_get(f"{JOLPICA}/{YEAR}/{rnd}/sprint.json", params={"limit": 30})
            rd = resp["MRData"]["RaceTable"]["Races"]
            if rd and "SprintResults" in rd[0]:
                for r in rd[0]["SprintResults"]:
                    all_sprints.append({"round": int(rnd), "race_name": name, "driverId": r["Driver"]["driverId"],
                                        "driverCode": r["Driver"].get("code",""), "constructorId": r["Constructor"]["constructorId"],
                                        "grid": int(r["grid"]), "position": r.get("position"),
                                        "points": float(r.get("points",0)), "laps": int(r.get("laps",0))})
        except: pass
        time.sleep(0.3)
    if all_sprints: pd.DataFrame(all_sprints).to_csv(f"{DATA_DIR}/sprints_{YEAR}.csv", index=False)
    print(f"   Done: {len(all_sprints)}")
time.sleep(0.5)

# 9. OpenF1 stints + drivers
print("9. OpenF1 stints and drivers...")
stints_done = already_done(f"stints_openf1_{YEAR}.csv")
drivers_done = already_done(f"drivers_openf1_{YEAR}.csv")
if not stints_done or not drivers_done:
    sessions = requests.get(f"{OPENF1}/sessions", params={"year": YEAR, "session_type": "Race"}).json()
    time.sleep(1)
    all_drivers, all_stints = [], []
    for s in sessions:
        sk, name = s["session_key"], s.get("meeting_name", "")
        try:
            d = requests.get(f"{OPENF1}/drivers", params={"session_key": sk}).json()
            for drv in d: all_drivers.append({**drv, "session_key": sk, "race_name": name})
        except: pass
        time.sleep(1)
        for attempt in range(3):
            try:
                resp = requests.get(f"{OPENF1}/stints", params={"session_key": sk}).json()
                if isinstance(resp, list) and len(resp) > 0:
                    for st in resp: all_stints.append({**st, "session_key": sk, "race_name": name})
                    print(f"   [{sk}] {name} done ({len(resp)} stints)")
                    break
            except: pass
            time.sleep(2)
        time.sleep(1.5)
    pd.DataFrame(all_drivers).to_csv(f"{DATA_DIR}/drivers_openf1_{YEAR}.csv", index=False)
    pd.DataFrame(all_stints).to_csv(f"{DATA_DIR}/stints_openf1_{YEAR}.csv", index=False)
    print(f"   Done: {len(all_drivers)} drivers, {len(all_stints)} stints")

# 10. OpenF1 pit stops (stationary time + timestamps)
print("10. OpenF1 pit stops (timestamps + stationary time)...")
if not already_done(f"pitstops_openf1_{YEAR}.csv"):
    if 'sessions' not in dir():
        sessions = requests.get(f"{OPENF1}/sessions", params={"year": YEAR, "session_type": "Race"}).json()
        time.sleep(1)
    race_only = [s for s in sessions if "sprint" not in s.get("session_name", "").lower()]
    all_of1_pits = []
    for s in race_only:
        sk, name = s["session_key"], s.get("circuit_short_name", s.get("meeting_name", ""))
        try:
            resp = requests.get(f"{OPENF1}/pit", params={"session_key": sk}, timeout=30).json()
            if isinstance(resp, list):
                for p in resp:
                    all_of1_pits.append({
                        "session_key": sk,
                        "race_name": name,
                        "driver_number": int(p["driver_number"]),
                        "lap_number": int(p["lap_number"]),
                        "pit_duration": p.get("pit_duration"),
                        "date": p.get("date", ""),
                    })
                print(f"   [{sk}] {name}: {len(resp)} pit stops")
        except Exception as e:
            print(f"   [{sk}] {name}: ⚠ {e}")
        time.sleep(0.5)
    pd.DataFrame(all_of1_pits).to_csv(f"{DATA_DIR}/pitstops_openf1_{YEAR}.csv", index=False)
    print(f"   Done: {len(all_of1_pits)} pit records")
time.sleep(0.5)

# 11. OpenF1 position data (race position changes with timestamps)
print("11. OpenF1 position data (position timeline)...")
if not already_done(f"positions_openf1_{YEAR}.csv"):
    if 'sessions' not in dir():
        sessions = requests.get(f"{OPENF1}/sessions", params={"year": YEAR, "session_type": "Race"}).json()
        time.sleep(1)
    race_only = [s for s in sessions if "sprint" not in s.get("session_name", "").lower()]
    all_of1_pos = []
    for s in race_only:
        sk, name = s["session_key"], s.get("circuit_short_name", s.get("meeting_name", ""))
        try:
            resp = requests.get(f"{OPENF1}/position", params={"session_key": sk}, timeout=30).json()
            if isinstance(resp, list):
                for p in resp:
                    all_of1_pos.append({
                        "session_key": sk,
                        "driver_number": int(p["driver_number"]),
                        "position": int(p["position"]),
                        "date": p.get("date", ""),
                    })
                print(f"   [{sk}] {name}: {len(resp)} position records")
        except Exception as e:
            print(f"   [{sk}] {name}: ⚠ {e}")
        time.sleep(0.5)
    pd.DataFrame(all_of1_pos).to_csv(f"{DATA_DIR}/positions_openf1_{YEAR}.csv", index=False)
    print(f"   Done: {len(all_of1_pos)} position records")
time.sleep(0.5)

# 12. OpenF1 lap durations (for tyre degradation analysis)
print("12. OpenF1 lap durations...")
if not already_done(f"laps_openf1_{YEAR}.csv"):
    if 'sessions' not in dir():
        sessions = requests.get(f"{OPENF1}/sessions", params={"year": YEAR, "session_type": "Race"}).json()
        time.sleep(1)
    race_only = [s for s in sessions if "sprint" not in s.get("session_name", "").lower()]
    all_of1_laps = []
    for s in race_only:
        sk, name = s["session_key"], s.get("circuit_short_name", s.get("meeting_name", ""))
        try:
            resp = requests.get(f"{OPENF1}/laps", params={"session_key": sk}, timeout=30).json()
            if isinstance(resp, list):
                for lap in resp:
                    if lap.get("lap_duration") and lap["lap_duration"] > 0:
                        all_of1_laps.append({
                            "session_key": int(sk),
                            "driver_number": int(lap["driver_number"]),
                            "lap_number": int(lap["lap_number"]),
                            "lap_duration": float(lap["lap_duration"]),
                        })
                print(f"   [{sk}] {name}: {len([l for l in resp if l.get('lap_duration')])} laps")
        except Exception as e:
            print(f"   [{sk}] {name}: ⚠ {e}")
        time.sleep(0.5)
    pd.DataFrame(all_of1_laps).to_csv(f"{DATA_DIR}/laps_openf1_{YEAR}.csv", index=False)
    print(f"   Done: {len(all_of1_laps)} lap records")
time.sleep(0.5)

with open(MARKER, "w") as f: f.write("done")
print(f"\nAll data saved to Google Drive at:\n  {DATA_DIR}/")

1. Schedule...
   Already exists (24 rows) — skipping
2. Driver standings...
   Already exists (21 rows) — skipping
3. Constructor standings...
   Already exists (10 rows) — skipping
4. Race results...
   Already exists (479 rows) — skipping
5. Qualifying...
   Already exists (479 rows) — skipping
6. Pit stops...
   Already exists (821 rows) — skipping
7. Lap times (slowest step — ~5 min)...
   Already exists (3858 rows) — skipping
8. Sprints...
   Already exists (120 rows) — skipping
9. OpenF1 stints and drivers...
   Already exists (1433 rows) — skipping
   Already exists (579 rows) — skipping
10. OpenF1 pit stops (timestamps + stationary time)...
   Already exists (821 rows) — skipping
11. OpenF1 position data (position timeline)...
   Already exists (11564 rows) — skipping
12. OpenF1 lap durations...
   Already exists (26211 rows) — skipping

All data saved to Google Drive at:
  /content/drive/MyDrive/Big Data 2 - Ha, Bach Viet (Bob)/f1_data_2025/


In [6]:
# Load all DataFrames
schedule_df  = pd.read_csv(f"{DATA_DIR}/schedule_{YEAR}.csv")
results_df   = pd.read_csv(f"{DATA_DIR}/results_{YEAR}.csv")
quali_df     = pd.read_csv(f"{DATA_DIR}/qualifying_{YEAR}.csv")
pitstops_df  = pd.read_csv(f"{DATA_DIR}/pitstops_{YEAR}.csv")
laptimes_df  = pd.read_csv(f"{DATA_DIR}/laptimes_{YEAR}.csv")
drv_stand_df = pd.read_csv(f"{DATA_DIR}/driver_standings_{YEAR}.csv")
con_stand_df = pd.read_csv(f"{DATA_DIR}/constructor_standings_{YEAR}.csv")
stints_df    = pd.read_csv(f"{DATA_DIR}/stints_openf1_{YEAR}.csv")
drivers_df   = pd.read_csv(f"{DATA_DIR}/drivers_openf1_{YEAR}.csv")
sprints_path = f"{DATA_DIR}/sprints_{YEAR}.csv"
sprints_df   = pd.read_csv(sprints_path) if os.path.exists(sprints_path) else pd.DataFrame()

of1_pits_df  = pd.read_csv(f"{DATA_DIR}/pitstops_openf1_{YEAR}.csv")
of1_pos_df   = pd.read_csv(f"{DATA_DIR}/positions_openf1_{YEAR}.csv")
of1_laps_df  = pd.read_csv(f"{DATA_DIR}/laps_openf1_{YEAR}.csv")

print(f"All DataFrames loaded from: {DATA_DIR}/")
for name, df in [("Schedule", schedule_df), ("Results", results_df), ("Qualifying", quali_df),
                 ("Pit Stops", pitstops_df), ("Lap Times", laptimes_df), ("Stints", stints_df),
                 ("Drivers", drivers_df), ("Sprints", sprints_df),
                 ("OpenF1 Pits", of1_pits_df), ("OpenF1 Positions", of1_pos_df), ("OpenF1 Laps", of1_laps_df)]:
    print(f"  {name:20s} - {len(df)} rows")

All DataFrames loaded from: /content/drive/MyDrive/Big Data 2 - Ha, Bach Viet (Bob)/f1_data_2025/
  Schedule             - 24 rows
  Results              - 479 rows
  Qualifying           - 479 rows
  Pit Stops            - 821 rows
  Lap Times            - 3858 rows
  Stints               - 1433 rows
  Drivers              - 579 rows
  Sprints              - 120 rows
  OpenF1 Pits          - 821 rows
  OpenF1 Positions     - 11564 rows
  OpenF1 Laps          - 26211 rows


In [7]:
# Prepare stint data for top 4 teams
top_teams = ["Ferrari", "McLaren", "Mercedes", "Red Bull"]

stints = stints_df.copy()
drivers_lookup = drivers_df[["driver_number", "name_acronym", "team_name", "session_key"]].drop_duplicates()
stints = stints.merge(drivers_lookup, on=["driver_number", "session_key"], how="left")

team_map = {}
for tn in stints["team_name"].dropna().unique():
    for top in top_teams:
        if top.lower() in tn.lower():
            team_map[tn] = top
            break

stints["team_clean"] = stints["team_name"].map(team_map)
stints["stint_length"] = stints["lap_end"] - stints["lap_start"] + 1
top4 = stints[stints["team_clean"].isin(top_teams) & (stints["stint_length"] >= 3)].copy()

print(f"Stint data ready: {len(top4)} stints across {top4['race_name'].nunique()} races")
for team in top_teams:
    t = top4[top4["team_clean"] == team]
    print(f"  {team:12s}: {len(t):3d} stints | "
          f"S:{len(t[t['compound']=='SOFT']):2d}  M:{len(t[t['compound']=='MEDIUM']):2d}  "
          f"H:{len(t[t['compound']=='HARD']):2d}  I:{len(t[t['compound']=='INTERMEDIATE']):2d}")

Stint data ready: 486 stints across 0 races
  Ferrari     : 122 stints | S:15  M:60  H:40  I: 7
  McLaren     : 121 stints | S:19  M:55  H:39  I: 8
  Mercedes    : 122 stints | S:22  M:53  H:39  I: 8
  Red Bull    : 121 stints | S:25  M:55  H:34  I: 7


In [8]:
# Reusable graph style
GRAPH_STYLE = dict(
    plot_bgcolor="#0f0f0f",
    paper_bgcolor="#0f0f0f",
    font=dict(color="#E8E8E8", family="Inter, sans-serif", size=14),
    height=650,
)

# Download logos and colorize them with team colors
import requests
import base64
from io import BytesIO
from PIL import Image

def load_and_colorize(url, hex_color):
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        img = Image.open(BytesIO(resp.content)).convert("RGBA")
        r, g, b = int(hex_color[1:3], 16), int(hex_color[3:5], 16), int(hex_color[5:7], 16)
        pixels = img.load()
        for y in range(img.height):
            for x in range(img.width):
                pr, pg, pb, pa = pixels[x, y]
                if pa > 0:
                    brightness = max(pr, pg, pb) / 255
                    pixels[x, y] = (int(r * brightness), int(g * brightness), int(b * brightness), pa)
        buf = BytesIO()
        img.save(buf, format="PNG")
        b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
        return f"data:image/png;base64,{b64}"
    except Exception as e:
        print(f"  Error: {e}")
        return None

def load_image_b64(url):
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        b64 = base64.b64encode(resp.content).decode("utf-8")
        content_type = resp.headers.get("Content-Type", "image/png")
        return f"data:{content_type};base64,{b64}"
    except Exception as e:
        print(f"  Error: {e}")
        return None

# Logo URLs (white originals from F1 CDN)
logo_urls = {
    "Ferrari": "https://media.formula1.com/image/upload/c_lfill,w_200/q_auto,f_png/v1740000001/common/f1/2026/ferrari/2026ferrarilogowhite.webp",
    "McLaren": "https://media.formula1.com/image/upload/c_lfill,w_200/q_auto,f_png/v1740000001/common/f1/2026/mclaren/2026mclarenlogowhite.webp",
    "Mercedes": "https://media.formula1.com/image/upload/c_lfill,w_200/q_auto,f_png/v1740000001/common/f1/2026/mercedes/2026mercedeslogowhite.webp",
    "Red Bull": "https://media.formula1.com/image/upload/c_lfill,w_200/q_auto,f_png/v1740000001/common/f1/2026/redbullracing/2026redbullracinglogowhite.webp",
}

# Car URLs (F1 CDN)
car_urls = {
    "Ferrari": "https://media.formula1.com/image/upload/c_lfill,w_512/q_auto,f_png/v1740000001/common/f1/2026/ferrari/2026ferraricarright.webp",
    "McLaren": "https://media.formula1.com/image/upload/c_lfill,w_512/q_auto,f_png/v1740000001/common/f1/2026/mclaren/2026mclarencarright.webp",
    "Mercedes": "https://media.formula1.com/image/upload/c_lfill,w_512/q_auto,f_png/v1740000001/common/f1/2026/mercedes/2026mercedescarright.webp",
    "Red Bull": "https://media.formula1.com/image/upload/c_lfill,w_512/q_auto,f_png/v1740000001/common/f1/2026/redbullracing/2026redbullracingcarright.webp",
}

# Colorize logos with team colors
print("Loading colored team logos...")
TEAM_LOGOS = {}
for team, url in logo_urls.items():
    img = load_and_colorize(url, TEAM_COLORS[team])
    if img:
        TEAM_LOGOS[team] = img
        print(f"  {team}: loaded + colorized")
    else:
        print(f"  {team}: FAILED")

print("\nLoading team car images...")
TEAM_CARS = {}
for team, url in car_urls.items():
    img = load_image_b64(url)
    if img:
        TEAM_CARS[team] = img
        print(f"  {team}: loaded")
    else:
        print(f"  {team}: FAILED")

print("\nDone!")

def add_team_images(fig, teams, image_dict=None, y_position=-0.15, logo_size=0.12):
    if image_dict is None:
        image_dict = TEAM_LOGOS
    for i, team in enumerate(teams):
        if team in image_dict:
            fig.add_layout_image(dict(
                source=image_dict[team],
                x=i, y=y_position,
                xref="x", yref="paper",
                sizex=0.6, sizey=logo_size,
                xanchor="center", yanchor="top",
                layer="above",
            ))

def style_axis(fig, y_title, y_range=None, y_dtick=5):
    fig.update_xaxes(
        showgrid=False, showline=False,
        tickfont=dict(size=15, color="#bbb", family="Inter, sans-serif"),
    )
    fig.update_yaxes(
        title=dict(text=y_title, font=dict(size=14, color="#777", family="Inter, sans-serif")),
        showgrid=True, gridcolor="rgba(255,255,255,0.06)", gridwidth=1,
        dtick=y_dtick, tickfont=dict(size=12, color="#555"),
        zeroline=False, showline=False,
        range=y_range,
    )

Loading colored team logos...
  Ferrari: loaded + colorized
  McLaren: loaded + colorized
  Mercedes: loaded + colorized
  Red Bull: loaded + colorized

Loading team car images...
  Ferrari: loaded
  McLaren: loaded
  Mercedes: loaded
  Red Bull: loaded

Done!


# Tire Compound and Degradation

In [20]:
# GRAPH 1: Compound Usage Profile (Dry Only)

dry = top4[top4["compound"].isin(["SOFT", "MEDIUM", "HARD"])].copy()
teams_order = ["Ferrari", "McLaren", "Mercedes", "Red Bull"]

compound_laps = dry.groupby(["team_clean", "compound"])["stint_length"].sum().reset_index(name="total_laps")
team_totals = compound_laps.groupby("team_clean")["total_laps"].sum().rename("team_total")
compound_laps = compound_laps.merge(team_totals, on="team_clean")
compound_laps["pct"] = compound_laps["total_laps"] / compound_laps["team_total"] * 100

fig = go.Figure()

for compound in ["SOFT", "MEDIUM", "HARD"]:
    vals, texts = [], []
    for team in teams_order:
        row = compound_laps[(compound_laps["team_clean"] == team) & (compound_laps["compound"] == compound)]
        if len(row) > 0:
            pct, laps = row["pct"].values[0], int(row["total_laps"].values[0])
            vals.append(pct)
            texts.append(f"<b>{pct:.0f}%</b><br><span style='font-size:11px'>{laps} laps</span>")
        else:
            vals.append(0)
            texts.append("")

    fig.add_trace(go.Bar(
        name=compound, x=teams_order, y=vals,
        text=texts, textposition="inside",
        textfont=dict(size=14, color="#111", family="Inter, sans-serif"),
        marker=dict(color=COMPOUND_COLORS[compound],
                    line=dict(color="rgba(0,0,0,0.4)", width=0.8)),
        width=0.55,
    ))

# Team accent lines
for i, team in enumerate(teams_order):
    fig.add_shape(type="rect", x0=i-0.3, x1=i+0.3, y0=-2, y1=-0.5,
                  fillcolor=TEAM_COLORS[team], line=dict(width=0), layer="above")

# Team logos
add_team_images(fig, teams_order, TEAM_LOGOS, y_position=-0.06, logo_size=0.14)

fig.update_layout(
    **GRAPH_STYLE,
    barmode="stack",
    title=dict(
        text=(f"<b>TYRE COMPOUND USAGE</b><br>"
              f"<span style='font-size:13px;color:#666;font-weight:normal'>"
              f"{YEAR} Season — Dry compounds only — % of total race laps</span>"),
        font=dict(size=24, color="#E8E8E8", family="Inter, sans-serif"),
        x=0.5, xanchor="center", y=0.95,
    ),
    legend=dict(
        orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5,
        font=dict(size=14, family="Inter, sans-serif"), bgcolor="rgba(0,0,0,0)",
    ),
    margin=dict(l=65, r=30, t=130, b=140),
)

style_axis(fig, y_title="Share of Race Laps (%)", y_range=[0, 108], y_dtick=25)

fig.show()

This stacked bar chart displays the percentage share of total race laps completed on each dry tire compound (soft, medium, hard) by the four leading teams, Ferrari, McLaren, Mercedes, and Red Bull, during the 2025 season. The x-axis represents each team, while the y-axis shows the proportion of race laps completed on each compound.

The graph reveals clear differences in compound usage between the four teams. Ferrari recorded the lowest soft compound usage at approximately 10%, while Red Bull used the highest proportion of soft laps at 17%. Ferrari also completed the highest percentage of laps on the medium compound at 49%. Mercedes relied most heavily on the hard compound, with 43% of its race laps completed on hards, the highest among the teams.

Ferrari’s distribution suggests a preference for longer and more stable race stints, with a strong focus on the medium compound and limited use of the soft tire. Medium tires generally provide a balance between pace and durability, so Ferrari’s strategy appears to prioritise consistency and reducing tire degradation over taking higher risks on softer compounds.

In contrast, Red Bull showed a more aggressive compound distribution, using the soft tire more frequently and the hard tire the least out of the four teams. Soft tires provide the highest grip and fastest lap times, especially at the beginning of a stint, but they wear out more quickly. This can indicate that Red Bull often prioritised gaining positions early in races, attacking during shorter stints, or maximising pace when trying to close gaps to rivals such as McLaren during the season. Their higher soft usage suggests a greater willingness to trade long term tire life for short term speed advantages.

Mercedes displayed the most balanced split between medium and hard compounds, with both making up over 40% of total laps. Hard tires are slower initially but last much longer, so Mercedes’ distribution suggests a strategy focused on extending stints and maintaining consistent race pace over longer periods. Their relatively low soft compound usage also indicates less reliance on aggressive short stints and more emphasis on stability across race distance.

McLaren’s compound usage sits between the other teams across all three compounds, without an extreme reliance on any single tire type. This balanced distribution suggests the team was comfortable adapting strategies depending on race situations. Compared to Red Bull’s more aggressive soft tire usage, McLaren’s lower soft percentage may indicate a preference for maintaining their pole position and managing races consistently rather than relying heavily on high risk attacking strategies.

In [9]:
# GRAPH 2: Median Stint Length by Compound

dry = top4[top4["compound"].isin(["SOFT", "MEDIUM", "HARD"])].copy()
teams_order = ["Ferrari", "McLaren", "Mercedes", "Red Bull"]

fig = go.Figure()

for compound in ["SOFT", "MEDIUM", "HARD"]:
    medians, q1s, q3s = [], [], []
    for team in teams_order:
        subset = dry[(dry["team_clean"] == team) & (dry["compound"] == compound)]["stint_length"]
        med = subset.median()
        medians.append(med)
        q1s.append(med - subset.quantile(0.25))
        q3s.append(subset.quantile(0.75) - med)

    fig.add_trace(go.Bar(
        name=compound, x=teams_order, y=medians,
        error_y=dict(type="data", symmetric=False, array=q3s, arrayminus=q1s,
                     color="rgba(255,255,255,0.3)", thickness=1.5, width=5),
        marker=dict(color=COMPOUND_COLORS[compound],
                    line=dict(color="rgba(0,0,0,0.4)", width=0.8)),
        text=[f"<b>{m:.0f}</b>" for m in medians],
        textposition="outside",
        textfont=dict(size=14, color="#bbb", family="Inter, sans-serif"),
        width=0.22,
        hovertemplate=f"<b>%{{x}} — {compound}</b><br>Median: %{{y:.1f}} laps<extra></extra>",
    ))

# Team accent lines
for i, team in enumerate(teams_order):
    fig.add_shape(type="rect", x0=i-0.38, x1=i+0.38, y0=-0.8, y1=0,
                  fillcolor=TEAM_COLORS[team], line=dict(width=0), layer="above",
                  opacity=0.7)

# Team logos
add_team_images(fig, teams_order, TEAM_LOGOS, y_position=-0.06, logo_size=0.14)

fig.update_layout(
    **GRAPH_STYLE,
    barmode="group",
    title=dict(
        text=(f"<b>STINT LENGTH BY COMPOUND</b><br>"
              f"<span style='font-size:13px;color:#666;font-weight:normal'>"
              f"{YEAR} Season — Median laps per tyre stint (error bars = Q1–Q3 range)</span>"),
        font=dict(size=24, color="#E8E8E8", family="Inter, sans-serif"),
        x=0.5, xanchor="center", y=0.95,
    ),
    legend=dict(
        orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5,
        font=dict(size=14, family="Inter, sans-serif"), bgcolor="rgba(0,0,0,0)",
    ),
    margin=dict(l=65, r=30, t=130, b=140),
)

style_axis(fig, y_title="Median Stint Length (laps)", y_range=[0, 42], y_dtick=5)

fig.show()

This bar chart compares the median number of laps each team completed per stint on the soft, medium, and hard compounds during the 2025 season. The error bars represent the interquartile range (Q1 to Q3), showing the variation in stint lengths across races. The x-axis lists the four teams, while the y-axis measures median stint length in laps.

Ferrari recorded the shortest median stints on both the soft and hard compounds, with 16 laps on softs and 25 on hards. On mediums, however, Ferrari's median of 20 laps was the second highest among the four teams, above both Mercedes and Red Bull at 19 laps each. This is notable because Ferrari also used the medium compound more frequently than any other team in Graph 1. Despite favouring more durable compounds, Ferrari still completed shorter stints on hards than its rivals, suggesting the team often chose to pit earlier on the harder tyres rather than extend tyre life.

McLaren achieved some of the longest stints overall, particularly on the hard compound with a median of 29 laps. Their medium stint median of 21 laps was also the highest among the teams. The difference between their soft and hard stint lengths was the largest in the field at 12 laps, showing that McLaren extracted significantly more life when switching to harder compounds. This suggests the team was able to maximise tyre durability during long runs, giving them greater flexibility in strategy and allowing them to reduce pit stop pressure during races. Another reason for McLaren's longer stints is to avoid pitstops to protect their pole positions.

Mercedes showed a unique distribution, with a median soft stint of 20 laps, slightly longer than its 19-lap median on mediums. Although unusual, this indicates that Mercedes was sometimes able to extend soft tyre stints to lengths comparable with medium tyres. Their hard compound median of 29 laps matched McLaren for the longest in the field, reinforcing the trend from Graph 1 where Mercedes also completed the highest percentage of race laps on hard tyres. Overall, Mercedes appears to have relied heavily on long and stable stints, particularly on harder compounds.

Red Bull's stint lengths placed them in the middle range across all compounds, with 17 laps on softs, 19 on mediums, and 26 laps on hards. Combined with Graph 1, where Red Bull used the soft compound more than any other team, this suggests the team frequently prioritised shorter and faster stints rather than maximising tyre longevity. Their higher soft usage, together with only moderate hard stint lengths, indicates a strategy focused more on pace and attacking opportunities than extending tyre life over long runs.

Across all four teams, the expected trend is visible, with harder compounds producing longer stints than softer compounds. This difference is strategically important because teams that can significantly extend hard tyre stints have more flexibility when responding to safety cars, pit stop timing, and changing race conditions.

In [12]:
# Graph 3: Tyre Degradation on Different Compounds

import numpy as np

teams_order = ["Ferrari", "McLaren", "Mercedes", "Red Bull"]
compounds   = ["SOFT", "MEDIUM", "HARD"]
COMP_COLORS = {"SOFT": "#FF3333", "MEDIUM": "#FFC800", "HARD": "#CCCCCC"}

dry = top4[top4["compound"].isin(compounds)].copy()

# Build degradation data from CSV
all_curves = []
matched, skipped = 0, 0

for _, stint in dry.iterrows():
    sk = int(stint["session_key"])
    dn = int(stint["driver_number"]) if pd.notna(stint["driver_number"]) else None
    if not dn: skipped += 1; continue

    lap_s, lap_e = int(stint["lap_start"]) + 2, int(stint["lap_end"])
    if lap_e - lap_s < 2: skipped += 1; continue

    laps = of1_laps_df[
        (of1_laps_df["session_key"]   == sk) &
        (of1_laps_df["driver_number"] == dn) &
        (of1_laps_df["lap_number"]    >= lap_s) &
        (of1_laps_df["lap_number"]    <= lap_e)
    ].sort_values("lap_number").copy()
    if len(laps) < 3: skipped += 1; continue

    med  = laps["lap_duration"].median()
    laps = laps[(laps["lap_duration"] <= med * 1.05) & (laps["lap_duration"] >= med * 0.97)]
    if len(laps) < 3: skipped += 1; continue

    baseline = laps["lap_duration"].iloc[:2].mean()
    laps["tyre_age"] = range(1, len(laps) + 1)
    laps["deg"]      = laps["lap_duration"] - baseline
    laps["compound"] = stint["compound"]
    laps["team"]     = stint["team_clean"]
    all_curves.append(laps[["tyre_age", "deg", "compound", "team"]])
    matched += 1

deg_df = pd.concat(all_curves, ignore_index=True)

# Average, smooth, normalize
avg = (deg_df.groupby(["team", "compound", "tyre_age"])["deg"]
       .agg(["mean", "count"]).reset_index())
avg = avg[avg["count"] >= 2].sort_values(["team", "compound", "tyre_age"])
avg["smooth"] = (avg.groupby(["team", "compound"])["mean"]
                 .transform(lambda x: x.rolling(3, min_periods=1, center=True).mean()))

for team in teams_order:
    for compound in compounds:
        mask = (avg["team"] == team) & (avg["compound"] == compound)
        sub  = avg[mask].sort_values("tyre_age")
        if len(sub) > 0:
            avg.loc[mask, "smooth"] -= sub["smooth"].iloc[0]

# Display range
compound_team_max = {}
for compound in compounds:
    team_maxes = [
        avg[(avg["team"] == t) & (avg["compound"] == compound)]["tyre_age"].max()
        for t in teams_order
        if len(avg[(avg["team"] == t) & (avg["compound"] == compound)]) >= 3
    ]
    if team_maxes:
        compound_team_max[compound] = min(team_maxes)

DISPLAY_MAX = min(max(compound_team_max.values(), default=20), 28)
avg = avg[avg["tyre_age"] <= DISPLAY_MAX]

# Y-axis range
g_min = avg["smooth"].min()
g_max = avg["smooth"].max()
abs_m = max(abs(g_min), abs(g_max))
pad   = abs_m * 0.25
YMIN  = round(g_min - pad, 2)
YMAX  = round(g_max + pad, 2)
if YMIN > -abs_m * 0.10:
    YMIN = round(-abs_m * 0.10 - pad * 0.5, 2)
if YMAX < abs_m * 0.10:
    YMAX = round( abs_m * 0.10 + pad * 0.5, 2)

# Compute slopes
team_slopes = {}
for team in teams_order:
    team_slopes[team] = {}
    for compound in compounds:
        sub = avg[(avg["team"] == team) & (avg["compound"] == compound)].sort_values("tyre_age")
        if len(sub) >= 3:
            team_slopes[team][compound] = np.polyfit(sub["tyre_age"], sub["smooth"], 1)[0]

# Reusable chart builder
def make_deg_chart(compound):
    cc = COMP_COLORS[compound]
    fig = go.Figure()

    for ti, team in enumerate(teams_order):
        hc = TEAM_COLORS[team]
        r, g, b = int(hc[1:3], 16), int(hc[3:5], 16), int(hc[5:7], 16)
        sub = avg[(avg["team"] == team) & (avg["compound"] == compound)].sort_values("tyre_age")
        if len(sub) < 3:
            continue

        slope = team_slopes[team].get(compound, 0)
        end_y = float(sub["smooth"].iloc[-1])
        end_x = float(sub["tyre_age"].iloc[-1])

        # Fill area
        fig.add_trace(go.Scatter(
            x=pd.concat([sub["tyre_age"], sub["tyre_age"][::-1]]),
            y=pd.concat([sub["smooth"], pd.Series([0]*len(sub))]),
            fill="toself",
            fillcolor=f"rgba({r},{g},{b},0.05)",
            line=dict(width=0),
            showlegend=False, hoverinfo="skip",
        ))

        # Main trend line
        fig.add_trace(go.Scatter(
            x=sub["tyre_age"],
            y=sub["smooth"],
            mode="lines",
            line=dict(color=hc, width=4.5, shape="spline", smoothing=1.2),
            name=team,
            showlegend=True,
            legendgroup=team,
            hovertemplate=(
                f"<b>{team}</b> · Lap %{{x:.0f}}<br>"
                "Delta <b>%{y:+.3f}s</b><br>"
                f"Deg rate <b>{slope:+.3f} s/lap</b>"
                "<extra></extra>"
            ),
        ))

        # Endpoint dot + slope label
        fig.add_trace(go.Scatter(
            x=[end_x], y=[end_y],
            mode="markers+text",
            marker=dict(color=hc, size=10, line=dict(color="#0f0f0f", width=2)),
            text=[f"  {slope:+.3f} s/lap"],
            textposition="middle right",
            textfont=dict(size=11, color=hc, family="Inter, sans-serif"),
            showlegend=False, hoverinfo="skip",
        ))

    fig.add_hline(y=0, line=dict(color="rgba(255,255,255,0.08)", width=1), layer="below")

    fig.add_annotation(
        text="SLOWER (DEGRADING)", x=0.005, y=0.98,
        xref="paper", yref="paper", showarrow=False,
        font=dict(size=9, color="rgba(248,113,113,0.35)", family="Inter, sans-serif"),
        xanchor="left", yanchor="top",
    )
    fig.add_annotation(
        text="FASTER (FUEL BURN-OFF)", x=0.005, y=0.02,
        xref="paper", yref="paper", showarrow=False,
        font=dict(size=9, color="rgba(74,222,128,0.35)", family="Inter, sans-serif"),
        xanchor="left", yanchor="bottom",
    )

    fig.update_layout(
        **GRAPH_STYLE,
        title=dict(
            text=f"<b>TYRE DEGRADATION</b>  <span style='color:{cc};font-size:18px'>{compound}</span>",
            font=dict(size=24, color="#E8E8E8", family="Inter, sans-serif"),
            x=0.5, xanchor="center", y=0.95,
        ),
        margin=dict(l=65, r=80, t=110, b=90),
        hovermode="x unified",
        hoverlabel=dict(
            bgcolor="rgba(12,12,12,0.95)", bordercolor="#333",
            font=dict(size=12, family="Inter, sans-serif", color="#ddd"),
            namelength=0,
        ),
        legend=dict(
            orientation="h",
            yanchor="bottom", y=1.01,
            xanchor="center", x=0.5,
            font=dict(size=14, family="Inter, sans-serif", color="#bbb"),
            bgcolor="rgba(0,0,0,0)",
            itemsizing="constant",
            tracegroupgap=60,
        ),
    )

    fig.update_xaxes(
        showgrid=True, gridcolor="rgba(255,255,255,0.03)",
        showline=False, zeroline=False, dtick=5,
        range=[0.5, DISPLAY_MAX + 2],
        tickfont=dict(size=13, color="#bbb", family="Inter, sans-serif"),
        title=dict(text="Laps on Tyre", font=dict(size=13, color="#777", family="Inter, sans-serif"), standoff=10),
    )
    fig.update_yaxes(
        range=[YMIN, YMAX],
        showgrid=True, gridcolor="rgba(255,255,255,0.06)", gridwidth=1,
        dtick=0.25, tickformat="+.2f", ticksuffix="s",
        tickfont=dict(size=11, color="#555", family="Inter, sans-serif"),
        zeroline=False, showline=False,
        title=dict(text="Pace Delta", font=dict(size=13, color="#777", family="Inter, sans-serif"), standoff=14),
    )

    fig.show()

In [13]:
# GRAPH 3A: Soft tyre degradation
make_deg_chart("SOFT")

This line chart shows how each team’s pace changes over the course of a soft tyre stint during the 2025 season. The x-axis represents laps completed on the tyre, while the y-axis shows pace delta relative to the opening laps of the stint. Positive values indicate the car is becoming slower as the tyres degrade, while negative values indicate lap times are improving, often due to fuel burn off reducing the car’s weight. The slope value shown at the end of each line summarises the team’s average degradation rate in seconds per lap.

Ferrari recorded the highest soft compound degradation rate at +0.057 s/lap. The pace loss remains moderate during the early phase of the stint, reaching around +0.44 seconds by lap 10, but increases significantly in the later laps, finishing at approximately +1.52 seconds by lap 23. This shows Ferrari’s performance on the soft tyre declined more rapidly over longer stints compared to the other teams. The graph also supports the trends seen in Graph 1 and Graph 2, where Ferrari used the soft compound the least and completed the shortest soft stints. Together, these results suggest Ferrari avoided extending soft tyre runs because performance dropped more noticeably as the stint progressed.

Mercedes recorded the second highest degradation rate at +0.047 s/lap, ending at around +1.10 seconds by lap 22. The overall trend is similar to Ferrari, with degradation becoming more visible in the second half of the stint. However, Mercedes’ curve is smoother and more gradual, particularly between laps 10 and 15 where the pace delta remains relatively stable. This may help explain why Mercedes could still complete relatively long soft stints despite having a fairly high degradation slope overall. The tyres appear to maintain a usable performance window for longer before lap times begin to rise more significantly.

McLaren and Red Bull showed the lowest degradation rates, at +0.004 and +0.005 s/lap respectively. McLaren’s pace fluctuates within a narrow range throughout the stint and finishes slightly below its starting pace at around negative 0.16 seconds by lap 24. Red Bull shows a similarly stable pattern across the longest soft stint length in the graph, extending to 28 laps while remaining close to its starting pace.In both cases, the pace delta changes very little over the stint compared to Ferrari and Mercedes. This suggests both teams, McLaren and Redbull were able to maintain more consistent performance on the soft compound over longer runs.

The graph overall highlights a clear difference between the teams in soft tyre management. Ferrari and Mercedes experienced much larger increases in lap time over a stint, while McLaren and Red Bull maintained relatively stable pace across longer runs. This difference is reflected in their strategy choices from the earlier graphs, where McLaren and Red Bull were generally more willing to extend soft tyre usage, while Ferrari relied less on the compound and completed shorter soft stints.

In [14]:
# GRAPH 3B: Medium Tyre Degradation
make_deg_chart("MEDIUM")

Unlike the soft compound in Graph 3a, all four teams show near zero or negative degradation slopes on the medium tyre. This means that, overall, the cars either maintained their pace or became faster as the stint progressed. This suggests that the medium compound was generally more stable over long runs, with fuel burn off reducing lap times enough to offset most of the tyre wear.

Ferrari, McLaren, and Mercedes all recorded negative degradation slopes of -0.015, -0.021, and -0.017 s/lap respectively. Ferrari’s curve remains below the zero line for almost the entire stint and finishes around -0.84 seconds by lap 28, indicating that pace improved steadily across the run. McLaren follows a similar pattern and records the largest negative slope, ending around -0.75 seconds. This suggests McLaren maintained very consistent pace throughout medium stints while continuing to gain lap time as fuel load decreased.

Mercedes recorded the largest overall pace improvement, reaching approximately -1.03 seconds by lap 28. However, its curve fluctuates more noticeably than the other teams, with a noticable rise in degradation in lap 25. Despite this, the overall trend remains negative, showing that medium tyre performance stayed relatively stable across long runs.

Red Bull is the only team with a slightly positive slope at +0.007 s/lap. Although the curve occasionally drops below zero, it spends much of the stint slightly above the baseline and finishes close to its starting pace. Compared to Ferrari, McLaren, and Mercedes, Red Bull experienced less overall pace improvement during medium stints. However, the degradation level still remains relatively low, as the curve stays within a narrow range throughout the run.

Compared to Graph 3a, the medium compound produces much smaller performance losses for every team. Ferrari and Mercedes in particular show a major improvement relative to their soft tyre degradation trends, where both teams experienced much steeper pace drop off over longer stints. On the medium compound, all teams were able to maintain more stable pace over extended runs, which helps explain why mediums accounted for the largest proportion of race laps in Graph 1.

In [15]:
# GRAPH 3C: Hard Tyre Degradation
make_deg_chart("HARD")

All four teams recorded negative slopes on the hard compound, meaning each team became progressively faster over the course of the stint. Compared to the soft and medium compound graphs, the hard tyre shows the smallest overall degradation and the strongest influence from fuel burn off. As the cars became lighter during the stint, lap times generally improved faster than tyre wear reduced performance.

Mercedes recorded the steepest negative slope at -0.033 s/lap, finishing approximately 1.24 seconds faster by lap 28 than at the start of the stint. The curve declines steadily throughout the run and becomes even steeper in the final laps, showing continuous pace improvement across the stint. This matches the trends seen in Graph 1 and Graph 2, where Mercedes completed the highest proportion of laps on the hard compound and also recorded one of the longest hard stint lengths. The data suggests Mercedes was particularly effective at maintaining performance over long hard tyre runs.

Red Bull recorded the second strongest improvement at -0.018 s/lap, ending around -0.60 seconds by lap 28. Compared to their medium compound graph, where pace stayed closer to the starting baseline, the hard tyre appears to produce more stable long run performance. The curve remains relatively smooth throughout the stint, indicating consistent pace changes without major drops in performance.

McLaren finished at approximately -0.50 seconds with a slope of -0.012 s/lap. The curve stays within a narrow range for most of the stint, showing stable and controlled pace improvement across long runs. Unlike Mercedes, McLaren does not show a large late stint pace increase, but instead maintains a gradual and consistent trend throughout the run. This reflects the overall pattern seen in the previous graphs, where McLaren consistently showed stable tyre performance across all compounds without large swings in degradation.

Ferrari recorded the shallowest negative slope at -0.009 s/lap, ending around -0.45 seconds by lap 28. After an initial improvement in the early laps, the curve remains relatively flat through much of the middle section of the stint before dropping slightly again near the end. Compared to the other teams, Ferrari gained the least pace over a hard tyre run, which aligns with Graph 2 where Ferrari also recorded the shortest median hard stint length.

Across all three degradation graphs, the hard compound produced the most stable long run performance overall. Ferrari and Mercedes showed the largest improvement when moving from soft tyres to harder compounds, while McLaren remained consistently stable across every compound type. Red Bull displayed more variation between compounds, performing very strongly on softs and hards but showing less pace improvement on mediums.

# Pit Stop Efficiency

Lorem Ipsum is simply dummy text of the printing and typesetting industry. Lorem Ipsum has been the industry's standard dummy text ever since the 1500s, when an unknown printer took a galley of type and scrambled it to make a type specimen book. It has survived not only five centuries, but also the leap into electronic typesetting, remaining essentially unchanged. It was popularised in the 1960s with the release of Letraset sheets containing Lorem Ipsum passages, and more recently with desktop publishing software like Aldus PageMaker including versions of Lorem Ipsum.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  GRAPH 4 — PIT STOP DURATION DOT PLOT
#  Requires: cells 1-9 already run
# ══════════════════════════════════════════════════════════════════════

import requests as req
import numpy as np

teams_order = ["Ferrari", "McLaren", "Mercedes", "Red Bull"]

# ── Fetch OpenF1 stationary pit stop times ───────────────────────────
print("Fetching OpenF1 /pit data...")
race_sessions = req.get(
    f"{OPENF1}/sessions",
    params={"year": YEAR, "session_type": "Race"}, timeout=15
).json()
# Exclude sprints
race_sessions = [s for s in race_sessions if "sprint" not in s.get("session_name", "").lower()]

# Build session_key → race name lookup
sk_race_name = {}
for s in race_sessions:
    sk_race_name[s["session_key"]] = s.get("circuit_short_name", s.get("meeting_name", ""))

openf1_pits = []
for s in race_sessions:
    sk = s["session_key"]
    try:
        resp = req.get(f"{OPENF1}/pit", params={"session_key": sk}, timeout=15).json()
        if isinstance(resp, list):
            for p in resp:
                if p.get("pit_duration") and p["pit_duration"] > 0:
                    openf1_pits.append({
                        "session_key": sk,
                        "race_name": sk_race_name.get(sk, ""),
                        "driver_number": int(p["driver_number"]),
                        "lap_number": p.get("lap_number"),
                        "pit_duration": float(p["pit_duration"]),
                    })
    except:
        pass
    import time as _time; _time.sleep(0.3)

print(f"  {len(openf1_pits)} pit records")

# ── Build driver lookup: number → abbreviation ──────────────────────
drv_abbr = {}
drv_team_lookup = {}
for _, row in drivers_df.drop_duplicates(subset=["driver_number"]).iterrows():
    dn = int(row["driver_number"])
    drv_abbr[dn] = row.get("name_acronym", row.get("broadcast_name", str(dn)))
    drv_team_lookup[dn] = str(row.get("team_name", ""))

# ── Decide data source ──────────────────────────────────────────────
if len(openf1_pits) > 0:
    of1_box = pd.DataFrame(openf1_pits)
    is_stationary = of1_box["pit_duration"].median() < 10
else:
    of1_box = pd.DataFrame()
    is_stationary = False

if len(openf1_pits) > 0 and is_stationary:
    print("  Using stationary pit stop times")

    def match_team(dn):
        tn = drv_team_lookup.get(dn, "")
        for t in teams_order:
            if t.lower() in tn.lower(): return t
        return None

    of1_box["team"] = of1_box["driver_number"].apply(match_team)
    of1_box["driver"] = of1_box["driver_number"].map(drv_abbr)
    pits = of1_box[of1_box["team"].isin(teams_order)].copy()
    pits = pits.rename(columns={"pit_duration": "duration_sec"})

    # Add stop number per driver per race
    pits = pits.sort_values(["session_key", "driver_number", "lap_number"])
    pits["stop_num"] = pits.groupby(["session_key", "driver_number"]).cumcount() + 1

    y_label = "Stationary Time (s)"
else:
    print("  Using Jolpica pitlane transit times")
    def match_team(cn):
        cn = str(cn).lower()
        for t in teams_order:
            if t.lower() in cn: return t
        return None

    driver_team = results_df[["driverId", "constructorName"]].drop_duplicates()
    driver_team["team"] = driver_team["constructorName"].apply(match_team)
    dt_map = dict(zip(driver_team.dropna(subset=["team"])["driverId"],
                       driver_team.dropna(subset=["team"])["team"]))
    pits = pitstops_df.copy()
    pits["team"] = pits["driverId"].map(dt_map)
    if pits["duration_sec"].isna().all():
        def parse_dur(raw):
            try:
                raw = str(raw)
                if ":" in raw:
                    parts = raw.split(":")
                    return float(parts[0]) * 60 + float(parts[1])
                return float(raw)
            except:
                return np.nan
        pits["duration_sec"] = pits["duration_raw"].apply(parse_dur)
    pits = pits[pits["team"].isin(teams_order) & pits["duration_sec"].notna()].copy()
    pits["driver"] = pits.get("driverId", "")
    pits["race_name"] = pits.get("raceName", "")
    pits["stop_num"] = pits.groupby(["driverId", pits.get("round", pits.index)]).cumcount() + 1
    y_label = "Pitlane Time (s)"

print(f"  {len(pits)} stops for top-4 teams\n")

# ── Order by median (fastest first) ─────────────────────────────────
team_medians = []
for team in teams_order:
    t = pits[pits["team"] == team]["duration_sec"]
    if len(t) > 0:
        team_medians.append((team, t.median()))
team_medians.sort(key=lambda x: x[1])
ordered_teams = [t[0] for t in team_medians]

for rank, (team, med) in enumerate(team_medians, 1):
    t = pits[pits["team"] == team]["duration_sec"]
    print(f"  {rank}. {team:12s}: median {med:.2f}s | mean {t.mean():.2f}s | "
          f"best {t.min():.2f}s | worst {t.max():.2f}s ({len(t)} stops)")

# ══════════════════════════════════════════════════════════════════════
#  BUILD FIGURE
# ══════════════════════════════════════════════════════════════════════

fig = go.Figure()

for i, team in enumerate(ordered_teams):
    subset = pits[pits["team"] == team].copy()
    t = subset["duration_sec"].values
    hc = TEAM_COLORS[team]
    r, g, b = int(hc[1:3], 16), int(hc[3:5], 16), int(hc[5:7], 16)
    med = float(np.median(t))

    # Build customdata for rich hover
    # columns: [race_name, driver, stop_num, lap_number]
    race_names = subset["race_name"].fillna("").values if "race_name" in subset.columns else [""] * len(t)
    drivers = subset["driver"].fillna("").values if "driver" in subset.columns else [""] * len(t)
    stop_nums = subset["stop_num"].fillna(0).astype(int).values if "stop_num" in subset.columns else [0] * len(t)
    lap_nums = subset["lap_number"].fillna(0).astype(int).values if "lap_number" in subset.columns else [0] * len(t)
    customdata = np.column_stack([race_names, drivers, stop_nums, lap_nums])

    # Jitter dots horizontally
    np.random.seed(42 + i)
    jitter = np.random.uniform(-0.2, 0.2, size=len(t))

    fig.add_trace(go.Scatter(
        x=[i + j for j in jitter],
        y=t,
        mode="markers",
        marker=dict(
            color=f"rgba({r},{g},{b},0.5)",
            size=7,
            line=dict(color=f"rgba({r},{g},{b},0.8)", width=0.5),
        ),
        customdata=customdata,
        showlegend=False,
        hovertemplate=(
            f"<b>{team}</b><br>"
            "<b>%{customdata[1]}</b> · %{customdata[0]}<br>"
            "Stop %{customdata[2]} · Lap %{customdata[3]}<br>"
            "<b>%{y:.2f}s</b>"
            "<extra></extra>"
        ),
    ))

    # Median line
    fig.add_shape(
        type="line",
        x0=i - 0.28, x1=i + 0.28,
        y0=med, y1=med,
        line=dict(color=hc, width=2.5),
    )

    # Median label — positioned ABOVE the line with clear offset
    fig.add_annotation(
        x=i, y=med,
        text=f"<b>{med:.2f}s</b>",
        showarrow=True,
        arrowhead=0,
        arrowwidth=1,
        arrowcolor="rgba(255,255,255,0.15)",
        ax=0, ay=-28,
        font=dict(size=13, color="#eee", family="Inter, sans-serif"),
        bgcolor="rgba(15,15,15,0.8)",
        borderpad=3,
    )

# ── Team logos ──
add_team_images(fig, ordered_teams, TEAM_LOGOS, y_position=-0.06, logo_size=0.14)

# ── Layout ──
fig.update_layout(
    **GRAPH_STYLE,
    title=dict(
        text=f"<b>PIT STOP DURATION</b>",
        font=dict(size=24, color="#E8E8E8", family="Inter, sans-serif"),
        x=0.5, xanchor="center", y=0.95,
    ),
    margin=dict(l=65, r=30, t=100, b=140),
    xaxis=dict(
        tickmode="array",
        tickvals=list(range(len(ordered_teams))),
        ticktext=ordered_teams,
        showgrid=False,
        zeroline=False,
    ),
    yaxis=dict(
        zeroline=False,
    ),
)

style_axis(fig, y_title=y_label, y_dtick=5)

# Remove any default axis lines that could look like vertical lines
fig.update_xaxes(showline=False, zeroline=False)
fig.update_yaxes(showline=False, zeroline=False)

fig.show()

Fetching OpenF1 /pit data...
  821 pit records
  Using Jolpica pitlane transit times
  381 stops for top-4 teams

  1. Red Bull    : median 22.76s | mean 23.37s | best 12.94s | worst 40.15s (123 stops)
  2. Ferrari     : median 22.83s | mean 22.98s | best 14.13s | worst 40.08s (87 stops)
  3. McLaren     : median 23.04s | mean 23.09s | best 12.80s | worst 40.28s (83 stops)
  4. Mercedes    : median 23.09s | mean 22.64s | best 13.67s | worst 33.24s (88 stops)


Lorem Ipsum is simply dummy text of the printing and typesetting industry. Lorem Ipsum has been the industry's standard dummy text ever since the 1500s, when an unknown printer took a galley of type and scrambled it to make a type specimen book. It has survived not only five centuries, but also the leap into electronic typesetting, remaining essentially unchanged. It was popularised in the 1960s with the release of Letraset sheets containing Lorem Ipsum passages, and more recently with desktop publishing software like Aldus PageMaker including versions of Lorem Ipsum.

In [34]:
# GRAPH 5: Pit Stop Strategy — Number of Stops per Race
# Requires: cells 1-9 already run

teams_order = ["Ferrari", "McLaren", "Mercedes", "Red Bull"]

# ── Map drivers to teams ──
def match_team(cn):
    cn = str(cn).lower()
    if "ferrari" in cn: return "Ferrari"
    if "mclaren" in cn: return "McLaren"
    if "mercedes" in cn: return "Mercedes"
    if "red bull" in cn: return "Red Bull"
    return None

driver_team = results_df[["driverId", "constructorName"]].drop_duplicates().copy()
driver_team["team"] = driver_team["constructorName"].apply(match_team)
dt_map = dict(zip(driver_team.dropna(subset=["team"])["driverId"],
                   driver_team.dropna(subset=["team"])["team"]))

# ── Count stops per driver per race, then group by team ──
pits = pitstops_df.copy()
pits["team"] = pits["driverId"].map(dt_map)
pits = pits[pits["team"].isin(teams_order)].copy()

stops_per_entry = (pits.groupby(["round", "driverId", "team"])["stop"]
                   .max().reset_index().rename(columns={"stop": "num_stops"}))

# Cap at 3+ (rare 4-stoppers grouped with 3)
stops_per_entry["strategy"] = stops_per_entry["num_stops"].clip(upper=3)
stops_per_entry["strategy_label"] = stops_per_entry["strategy"].map(
    {1: "1-STOP", 2: "2-STOP", 3: "3+ STOP"}
)

# ── Build percentage data for stacked bar ──
strategy_order = ["1-STOP", "2-STOP", "3+ STOP"]
STRATEGY_COLORS = {
    "1-STOP":   "rgb(80, 200, 120)",   # green — conservative/efficient
    "2-STOP":   "rgb(255, 180, 50)",    # amber — standard
    "3+ STOP":  "rgb(255, 65, 65)",     # red — aggressive/forced
}

fig = go.Figure()

for strat in strategy_order:
    vals, texts = [], []
    for team in teams_order:
        t = stops_per_entry[stops_per_entry["team"] == team]
        total = len(t)
        count = len(t[t["strategy_label"] == strat])
        pct = count / total * 100 if total > 0 else 0
        vals.append(pct)
        texts.append(f"<b>{pct:.0f}%</b><br><span style='font-size:11px'>{count} races</span>")

    fig.add_trace(go.Bar(
        name=strat, x=teams_order, y=vals,
        text=texts, textposition="inside",
        textfont=dict(size=14, color="#111", family="Inter, sans-serif"),
        marker=dict(color=STRATEGY_COLORS[strat],
                    line=dict(color="rgba(0,0,0,0.4)", width=0.8)),
        width=0.55,
        hovertemplate=f"<b>%{{x}} — {strat}</b><br>%{{y:.1f}}% of races<extra></extra>",
    ))

# ── Team accent bars ──
for i, team in enumerate(teams_order):
    fig.add_shape(type="rect", x0=i-0.3, x1=i+0.3, y0=-2, y1=-0.5,
                  fillcolor=TEAM_COLORS[team], line=dict(width=0), layer="above")

# ── Team logos ──
add_team_images(fig, teams_order, TEAM_LOGOS, y_position=-0.06, logo_size=0.14)

# ── Layout ──
fig.update_layout(
    **GRAPH_STYLE,
    barmode="stack",
    title=dict(
        text=(
            f"<b>PIT STOP STRATEGY</b><br>"
            f"<span style='font-size:13px;color:#666;font-weight:normal'>"
            f"{YEAR} Season — % of races by number of pit stops</span>"
        ),
        font=dict(size=24, color="#E8E8E8", family="Inter, sans-serif"),
        x=0.5, xanchor="center", y=0.95,
    ),
    legend=dict(
        orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5,
        font=dict(size=14, family="Inter, sans-serif"), bgcolor="rgba(0,0,0,0)",
    ),
    margin=dict(l=65, r=30, t=130, b=140),
)

style_axis(fig, y_title="Share of Races (%)", y_range=[0, 108], y_dtick=25)

fig.show()

Lorem Ipsum is simply dummy text of the printing and typesetting industry. Lorem Ipsum has been the industry's standard dummy text ever since the 1500s, when an unknown printer took a galley of type and scrambled it to make a type specimen book. It has survived not only five centuries, but also the leap into electronic typesetting, remaining essentially unchanged. It was popularised in the 1960s with the release of Letraset sheets containing Lorem Ipsum passages, and more recently with desktop publishing software like Aldus PageMaker including versions of Lorem Ipsum.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  GRAPH 6 — POSITION CHANGE THROUGH PIT STOPS
#  Measures: position at pit entry vs position 5 laps after
#  Data:     of1_pos_df, of1_pits_df, drivers_df (from CSVs)
#  Requires: cells 1-9 already run
# ══════════════════════════════════════════════════════════════════════

import numpy as np, warnings
warnings.filterwarnings("ignore", message="no explicit representation of timezones")

teams_order = ["Ferrari", "McLaren", "Mercedes", "Red Bull"]
LAPS_AFTER  = 5

# ── Validate position data ───────────────────────────────────────────
if len(of1_pos_df) < 5000:
    print("⚠ Position data too small — re-downloading...")
    import requests as req, time as _time
    sessions = req.get(f"{OPENF1}/sessions",
                       params={"year": YEAR, "session_type": "Race"}, timeout=15).json()
    race_only = [s for s in sessions if "sprint" not in s.get("session_name", "").lower()]
    all_of1_pos = []
    for s in race_only:
        sk = s["session_key"]
        name = s.get("circuit_short_name", "")
        try:
            resp = req.get(f"{OPENF1}/position",
                           params={"session_key": int(sk)}, timeout=30).json()
            if isinstance(resp, list):
                for p in resp:
                    all_of1_pos.append({
                        "session_key": int(sk),
                        "driver_number": int(p["driver_number"]),
                        "position": int(p["position"]),
                        "date": p.get("date", ""),
                    })
        except:
            pass
        _time.sleep(0.3)
    of1_pos_df = pd.DataFrame(all_of1_pos)
    of1_pos_df.to_csv(f"{DATA_DIR}/positions_openf1_{YEAR}.csv", index=False)
    print(f"  ✓ {len(of1_pos_df)} records saved")

# ── Map driver_number → team ─────────────────────────────────────────
drv_team = {}
for _, row in drivers_df.drop_duplicates(subset=["driver_number", "team_name"]).iterrows():
    dn = int(row["driver_number"])
    tn = str(row.get("team_name", ""))
    for t in teams_order:
        if t.lower() in tn.lower():
            drv_team[dn] = t
            break

# ── Prepare data ─────────────────────────────────────────────────────
pit_df = of1_pits_df.copy()
pit_df["team"] = pit_df["driver_number"].map(drv_team)
pit_df = pit_df[pit_df["team"].isin(teams_order)].copy()
pit_df["date"] = pd.to_datetime(pit_df["date"], format="ISO8601", utc=True).dt.tz_localize(None)

pos_df = of1_pos_df.copy()
pos_df["date"] = pd.to_datetime(pos_df["date"], format="ISO8601", utc=True).dt.tz_localize(None)
pos_df = pos_df.sort_values("date")
print(f"Pit stops: {len(pit_df)} | Position records: {len(pos_df)}")

# ── Pre-index position data ──────────────────────────────────────────
pos_groups = {}
for (sk, dn), grp in pos_df.groupby(["session_key", "driver_number"]):
    dates = grp["date"].values.astype("datetime64[ns]")
    positions = grp["position"].values
    pos_groups[(int(sk), int(dn))] = (dates, positions)

def get_position_at_time(sk, dn, target_time):
    key = (int(sk), int(dn))
    if key not in pos_groups:
        return None
    dates, positions = pos_groups[key]
    target_np = np.datetime64(target_time)
    idx = np.searchsorted(dates, target_np, side="right") - 1
    return int(positions[idx]) if idx >= 0 else None

# ── Median lap time per session ──────────────────────────────────────
lt = laptimes_df.copy()
lt["time_sec"] = pd.to_numeric(lt["time_sec"], errors="coerce")
round_median = lt[lt["time_sec"].between(50, 200)].groupby("round")["time_sec"].median().to_dict()

session_round = {}
for sk in pit_df["session_key"].unique():
    sk_rows = of1_pits_df[of1_pits_df["session_key"] == sk]
    if len(sk_rows) > 0:
        race_name = str(sk_rows.iloc[0].get("race_name", "")).lower()
        for _, sr in schedule_df.iterrows():
            checks = [str(sr.get(c, "")).lower() for c in ["circuitId", "raceName", "locality", "country"]]
            if any(race_name in c or c in race_name for c in checks if len(c) > 2):
                session_round[sk] = int(sr["round"])
                break

FALLBACK_LAP = 90.0
avg_lap = {sk: round_median.get(session_round.get(sk), FALLBACK_LAP) for sk in pit_df["session_key"].unique()}

# ══════════════════════════════════════════════════════════════════════
#  Calculate position delta: pit entry → 5 laps after
# ══════════════════════════════════════════════════════════════════════

# Build session_key → race name lookup
sk_race_name = {}
for sk_val in of1_pits_df["session_key"].unique():
    rows = of1_pits_df[of1_pits_df["session_key"] == sk_val]
    sk_race_name[int(sk_val)] = str(rows.iloc[0].get("race_name", f"R{sk_val}"))

deltas = []
for _, pit in pit_df.iterrows():
    sk, dn, team = int(pit["session_key"]), int(pit["driver_number"]), pit["team"]
    pit_time = pit["date"]
    lap_sec = avg_lap.get(sk, FALLBACK_LAP)

    p_before = get_position_at_time(sk, dn, pit_time)
    if p_before is None:
        continue
    p_after = get_position_at_time(sk, dn, pit_time + pd.Timedelta(seconds=lap_sec * LAPS_AFTER))
    if p_after is None:
        continue

    deltas.append({"team": team, "pos_before": p_before, "pos_after": p_after,
                   "delta": p_before - p_after, "race": sk_race_name.get(sk, ""),
                   "pit_lap": int(pit["lap_number"])})

delta_df = pd.DataFrame(deltas)
print(f"Matched {len(delta_df)} pit stops\n")

for team in teams_order:
    t = delta_df[delta_df["team"] == team]
    if len(t):
        print(f"  {team:12s}: {len(t):3d} stops | avg {t['delta'].mean():+.2f} | "
              f"median {t['delta'].median():+.1f} | best {t['delta'].max():+d} | worst {t['delta'].min():+d}")

# ══════════════════════════════════════════════════════════════════════
#  BUILD FIGURE
# ══════════════════════════════════════════════════════════════════════

team_stats = []
for team in teams_order:
    t = delta_df[delta_df["team"] == team]
    if len(t) == 0:
        continue
    team_stats.append({
        "team": team,
        "avg_delta": t["delta"].mean(),
        "count": len(t),
        "gained": (t["delta"] > 0).sum(),
        "lost": (t["delta"] < 0).sum(),
        "neutral": (t["delta"] == 0).sum(),
    })

team_stats.sort(key=lambda x: x["avg_delta"], reverse=True)
ordered_teams = [s["team"] for s in team_stats]

fig = go.Figure()

for i, stat in enumerate(team_stats):
    team = stat["team"]
    hc = TEAM_COLORS[team]
    delta = stat["avg_delta"]
    r, g, b = int(hc[1:3], 16), int(hc[3:5], 16), int(hc[5:7], 16)

    # Build detailed hover: list races where positions gained/lost
    t = delta_df[delta_df["team"] == team].copy()
    gained = t[t["delta"] > 0].sort_values("delta", ascending=False)
    lost = t[t["delta"] < 0].sort_values("delta")

    hover_lines = [f"<b>{team}</b>", f"Avg position change: <b>{delta:+.2f}</b>",
                   f"{stat['count']} pit stops", ""]

    if len(gained) > 0:
        hover_lines.append(f"<span style='color:#4ade80'>▲ Gained ({len(gained)}):</span>")
        for _, row in gained.head(8).iterrows():
            hover_lines.append(
                f"  {row['race']} L{row['pit_lap']}: "
                f"P{row['pos_before']}→P{row['pos_after']} ({row['delta']:+d})")
        if len(gained) > 8:
            hover_lines.append(f"  ...and {len(gained)-8} more")

    if len(lost) > 0:
        hover_lines.append(f"<span style='color:#f87171'>▼ Lost ({len(lost)}):</span>")
        for _, row in lost.head(5).iterrows():
            hover_lines.append(
                f"  {row['race']} L{row['pit_lap']}: "
                f"P{row['pos_before']}→P{row['pos_after']} ({row['delta']:+d})")
        if len(lost) > 5:
            hover_lines.append(f"  ...and {len(lost)-5} more")

    fig.add_trace(go.Bar(
        x=[team], y=[delta],
        marker=dict(
            color=f"rgba({r},{g},{b},0.25)",
            line=dict(color=hc, width=2),
        ),
        width=0.5,
        showlegend=False,
        hovertemplate="<br>".join(hover_lines) + "<extra></extra>",
    ))

    # Delta value label
    fig.add_annotation(
        x=team, y=delta,
        text=f"<b>{delta:+.2f}</b>",
        showarrow=False,
        font=dict(size=18, color="#eee", family="Inter, sans-serif"),
        yanchor="bottom" if delta >= 0 else "top",
        yshift=8 if delta >= 0 else -8,
    )

    # Breakdown: gained / lost / same — more distance from bar top
    fig.add_annotation(
        x=team, y=delta,
        text=(f"<span style='color:#4ade80'>{stat['gained']} gained</span>"
              f"<span style='color:#333'>  ·  </span>"
              f"<span style='color:#f87171'>{stat['lost']} lost</span>"
              f"<span style='color:#333'>  ·  </span>"
              f"<span style='color:#666'>{stat['neutral']} same</span>"),
        showarrow=False,
        font=dict(size=10, family="Inter, sans-serif"),
        yanchor="bottom" if delta >= 0 else "top",
        yshift=35 if delta >= 0 else -35,
    )

# ── Zero line ──
fig.add_hline(y=0, line=dict(color="rgba(255,255,255,0.1)", width=1))

# ── Team logos ──
add_team_images(fig, ordered_teams, TEAM_LOGOS, y_position=-0.06, logo_size=0.14)

# ── Layout ──
fig.update_layout(
    **GRAPH_STYLE,
    title=dict(
        text=f"<b>POSITION CHANGE AFTER PIT STOPS ({LAPS_AFTER} LAPS)</b>",
        font=dict(size=24, color="#E8E8E8", family="Inter, sans-serif"),
        x=0.5, xanchor="center", y=0.95,
    ),
    margin=dict(l=65, r=30, t=100, b=140),
    xaxis=dict(categoryorder="array", categoryarray=ordered_teams),
)

style_axis(fig,
           y_title="Avg Positions Gained / Lost",
           y_dtick=0.5)

fig.show()

Pit stops: 382 | Position records: 11564
Matched 382 pit stops

  Ferrari     :  87 stops | avg +0.79 | median +0.0 | best +10 | worst -12
  McLaren     :  83 stops | avg +0.82 | median +0.0 | best +8 | worst -1
  Mercedes    :  88 stops | avg +1.35 | median +1.0 | best +6 | worst -5
  Red Bull    : 124 stops | avg +0.71 | median +0.0 | best +6 | worst -12


Lorem Ipsum is simply dummy text of the printing and typesetting industry. Lorem Ipsum has been the industry's standard dummy text ever since the 1500s, when an unknown printer took a galley of type and scrambled it to make a type specimen book. It has survived not only five centuries, but also the leap into electronic typesetting, remaining essentially unchanged. It was popularised in the 1960s with the release of Letraset sheets containing Lorem Ipsum passages, and more recently with desktop publishing software like Aldus PageMaker including versions of Lorem Ipsum.

# DRS and Overtakes

Lorem Ipsum is simply dummy text of the printing and typesetting industry. Lorem Ipsum has been the industry's standard dummy text ever since the 1500s, when an unknown printer took a galley of type and scrambled it to make a type specimen book. It has survived not only five centuries, but also the leap into electronic typesetting, remaining essentially unchanged. It was popularised in the 1960s with the release of Letraset sheets containing Lorem Ipsum passages, and more recently with desktop publishing software like Aldus PageMaker including versions of Lorem Ipsum.

# Straights and Corners

Lorem Ipsum is simply dummy text of the printing and typesetting industry. Lorem Ipsum has been the industry's standard dummy text ever since the 1500s, when an unknown printer took a galley of type and scrambled it to make a type specimen book. It has survived not only five centuries, but also the leap into electronic typesetting, remaining essentially unchanged. It was popularised in the 1960s with the release of Letraset sheets containing Lorem Ipsum passages, and more recently with desktop publishing software like Aldus PageMaker including versions of Lorem Ipsum.

# Driver Performances